# Constellation Detection — Full Pipeline (external images + multimodal + distinct-label)

This is the maximum-effort run. Select **Runtime → Change runtime type → GPU** (A100 best; T4/L4 work), then **Run all**. It:

1. Clones the repo (code + data) from GitHub.
2. Runs GPU wide-search patch matching to build candidate caches.
3. Trains the **multimodal** ensemble on downloaded, attributed **external ESO images** (two CNN patch encoders + candidate re-ranker), with whole scenes held out.
4. Runs the new **distinct-label** geometry (global one-to-one scene→constellation assignment).
5. Applies presence refinement, validates every CSV, and downloads all candidates.

### Honest expectation (read this)
A valid CSV does not predict a leaderboard score. This project has **never** reached 0.90+, and its own multimodal external-image experiment **failed its promotion gate** (it scored slightly worse than the classical 0.71544 on held-out scenes). Two structural limits cap what any method here can do: only **3 labelled scenes** exist to tune on, and roughly **8 of 71 labelled patches have no candidate at their true location** (geometry cannot place an undetected star). Realistically this run lands near **0.71544 ± ~0.05**, not 0.94. Upload the candidates, let Kaggle decide, and keep the best.

## 1. GPU check + clone repo

In [ ]:
import os, shutil, subprocess, sys
import torch
assert torch.cuda.is_available(), 'Enable a GPU runtime: Runtime > Change runtime type > GPU'
print('GPU:', torch.cuda.get_device_name(0))
REPO_URL = 'https://github.com/7dracoder/Constellation-Detection---CS-GY-6643.git'
ROOT = '/content/constellation'
shutil.rmtree(ROOT, ignore_errors=True)
subprocess.run(['git', 'clone', '--recurse-submodules', '-q', REPO_URL, ROOT], check=False)
if not os.path.isdir(ROOT):
    subprocess.run(['git', 'clone', '-q', REPO_URL, ROOT], check=True)
os.chdir(ROOT)
print('commit:', subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip())

In [ ]:
%pip install -q --upgrade scipy pillow opencv-python-headless

In [ ]:
from pathlib import Path
ROOTP = Path(ROOT)
required = ['train', 'validation', 'patterns', 'sample_submission.csv', 'train_ground_truth.csv',
            'constellation_pipeline.py', 'joint_geometric_solver.py', 'structural_refiner.py',
            'gpu_matcher.py', 'presence_refiner.py', 'multimodel_pipeline.py',
            'neural_patch_models.py', 'download_training_images.py', 'distinct_label_solver.py',
            'matcher_config_gpu_wide.json']
missing = [n for n in required if not (ROOTP / n).exists()]
assert not missing, f'Missing from repo (push them first): {missing}'
print('OK:', len(list((ROOTP/'validation').iterdir())), 'validation scenes,',
      len(list((ROOTP/'patterns').glob('*_pattern.png'))), 'patterns')

## 2. GPU patch matching → candidate caches (slow, GPU-bound)
Builds the top-16 candidate cache for validation and train scenes. Everything downstream reuses these. Also produces the classical baseline CSV.

In [ ]:
OUT = ROOTP / 'outputs'; OUT.mkdir(exist_ok=True)
CACHE_VAL = OUT / 'cache_validation'
CACHE_TRAIN = OUT / 'cache_train'
def run(*args):
    print('+', ' '.join(map(str, args)), flush=True)
    subprocess.run([str(a) for a in args], cwd=ROOT, check=True)

run(sys.executable, 'joint_geometric_solver.py', '--root', ROOT, '--config', 'matcher_config_gpu_wide.json',
    '--output', OUT/'submission_baseline.csv', '--split', 'validation', '--device', 'cuda',
    '--top-k', '16', '--graph-top-k', '3', '--proposals', '20000', '--cache-dir', CACHE_VAL,
    '--presence-mode', 'quantile', '--present-rate', '0.625', '--consensus-trials', '5')

run(sys.executable, 'joint_geometric_solver.py', '--root', ROOT, '--config', 'matcher_config_gpu_wide.json',
    '--output', OUT/'train_pred_baseline.csv', '--split', 'train', '--device', 'cuda',
    '--top-k', '16', '--graph-top-k', '3', '--proposals', '20000', '--cache-dir', CACHE_TRAIN,
    '--presence-mode', 'quantile', '--present-rate', '0.625')

## 3. Multimodal training on external ESO images
Downloads four attributed ESO observations, trains two CNN patch encoders + a candidate re-ranker, holds every scene out of the learned classifiers, and reports an honest promotion gate. Produces `submission_multimodel_experimental.csv` and `submission_recommended.csv` (which falls back to v4 if the ensemble fails its gate).

In [ ]:
run(sys.executable, '-u', 'multimodel_pipeline.py', '--root', ROOT,
    '--download', '--steps', '4000', '--proposals', '12000', '--trials', '3')

In [ ]:
import json
mm = OUT / 'multimodel'
report = json.loads((mm / 'report.json').read_text())
print('multimodal promoted:', report['promoted'])
print('held-out baseline total(loose):', round(report['baseline']['mean']['total_loose'], 4))
print('held-out ensemble total(loose):', round(report['challenger']['mean']['total_loose'], 4))
if not report['promoted']:
    print('NOTE: multimodal ensemble did NOT beat baseline on held-out scenes (as previously documented).')

## 4. Distinct-label geometry (the new identity lever)
Global one-to-one scene→constellation assignment on the same GPU candidates. Two variants: conservative (only reassign when competitive) and strict (all 16 distinct).

In [ ]:
run(sys.executable, 'distinct_label_solver.py', '--root', ROOT, '--config', 'matcher_config_gpu_wide.json',
    '--output', OUT/'submission_distinct.csv', '--split', 'validation', '--device', 'cpu',
    '--top-k', '16', '--graph-top-k', '3', '--proposals', '20000', '--cache-dir', CACHE_VAL,
    '--presence-mode', 'quantile', '--present-rate', '0.625', '--consensus-trials', '5',
    '--transform-model', 'similarity')

run(sys.executable, 'distinct_label_solver.py', '--root', ROOT, '--config', 'matcher_config_gpu_wide.json',
    '--output', OUT/'submission_distinct_strict.csv', '--split', 'validation', '--device', 'cpu',
    '--top-k', '16', '--graph-top-k', '3', '--proposals', '20000', '--cache-dir', CACHE_VAL,
    '--presence-mode', 'quantile', '--present-rate', '0.625', '--consensus-trials', '5',
    '--transform-model', 'similarity', '--strict-distinct')

## 5. Presence refinement + validate all candidates

In [ ]:
sources = ['submission_distinct', 'submission_distinct_strict', 'submission_baseline']
for name in sources:
    src = OUT / f'{name}.csv'
    dst = OUT / f'{name}_presence.csv'
    run(sys.executable, 'presence_refiner.py', '--root', ROOT, '--input', src, '--output', dst,
        '--split', 'validation', '--cache-dir', CACHE_VAL, '--train-cache-dir', CACHE_TRAIN)
    run(sys.executable, 'constellation_pipeline.py', 'validate', '--root', ROOT, '--output', dst)
# The multimodal experimental CSV is already presence-refined inside its pipeline.
run(sys.executable, 'constellation_pipeline.py', 'validate', '--root', ROOT,
    '--output', mm/'submission_multimodel_experimental.csv')

## 6. Compare candidates

In [ ]:
import csv as _csv
from collections import Counter
candidates = {
    'distinct+presence': OUT/'submission_distinct_presence.csv',
    'strict-distinct+presence': OUT/'submission_distinct_strict_presence.csv',
    'baseline+presence (0.715 family)': OUT/'submission_baseline_presence.csv',
    'multimodal experimental': mm/'submission_multimodel_experimental.csv',
}
for label, path in candidates.items():
    if not path.exists():
        print(f'{label:40} MISSING'); continue
    with path.open(newline='') as fh:
        rows = list(_csv.DictReader(fh))
    names = [r['constellation'] for r in rows]
    dup = {k: v for k, v in Counter(names).items() if v > 1 and k != 'unknown'}
    print(f'{label:40} unique={len(set(names)):2d}  dup={dup or "none"}  unknown={names.count("unknown")}')

## 7. Download everything
Upload to Kaggle in this order and keep the highest score:
1. `submission_distinct_presence.csv` (safest improvement)
2. `submission_distinct_strict_presence.csv` (if scenes are truly all distinct)
3. `submission_multimodel_experimental.csv` (external-image ensemble)
4. `submission_baseline_presence.csv` (safe 0.715-family fallback)

In [ ]:
from google.colab import files
for path in [OUT/'submission_distinct_presence.csv', OUT/'submission_distinct_strict_presence.csv',
             mm/'submission_multimodel_experimental.csv', OUT/'submission_baseline_presence.csv']:
    if path.exists():
        print('downloading', path.name)
        files.download(str(path))